# SpecialistTuner (LoRA) - Week 8 API Notebook
**Applied GenAI & Agentic AI Engineering Course · Week 8 · V2**

This notebook exercises every endpoint of the LoRA inference service (`w08v02c01`).
Each endpoint is shown two ways: **cURL (Windows cmd)** and **Python requests**.

---
### Before you start
1. Any machine - CPU is fine (Qwen3-0.6B) - with the stack installed: `pip install -r requirements.txt`
2. A trained LoRA adapter. The best checkpoint from the shipped run lives at `./out/checkpoint-82` - copy it to `./out/adapter` or pass the checkpoint path directly
3. `.env` filled in: `HF_TOKEN`, `HF_HUB_REPO`, `BASE_MODEL`
4. Server running: `uvicorn app.main:app --reload` (port 8000)
5. Run the **Setup** cell below once.

> **Windows note:** All curl cells use `%%cmd` with `\"` to escape inner quotes. Single quotes are not supported in Windows cmd.

In [ ]:
# Setup - run this cell first
import requests, json

BASE = 'http://localhost:8000'

# Demo message sent to the LoRA-tuned specialist model.
# ASCII only - used verbatim in %%cmd curl cells.
DEMO_NOTES = 'Why does the Transformer use multi-head attention instead of a single attention function?'

# Adapter path (relative to server cwd). Best shipped checkpoint = out/checkpoint-82;
# copy it to out/adapter or point straight at the checkpoint as done here.
ADAPTER = 'out/checkpoint-82'

print('Setup complete.')
print('BASE:', BASE)

---
## 1 · Health Check - `GET /health`
Confirms the server is alive and shows which model is loaded.

> This section is identical across all weeks. Do not modify it.

In [ ]:
%%cmd
curl -s http://localhost:8000/health

In [ ]:
# Health check - Python
r = requests.get(f'{BASE}/health')
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))

---
## 2 · LoRA Inference - `POST /v1/chat/completions`

Sends a chat message to the LoRA-tuned specialist model and returns the generated response.

Key concept: the server loads Qwen3-0.6B once via `@lru_cache` - full fp32 on CPU, 4-bit QLoRA when CUDA is present -
then wraps it with the saved PEFT adapter. Every request re-uses the cached model -
there is no per-request model load overhead.

Request body:
```json
{
  "messages": [{"role": "user", "content": "..."}],
  "max_new_tokens": 256,
  "temperature": 0.2
}
```

Query parameter: `adapter` (optional; omitted = base model, no adapter) - path to the PEFT adapter/checkpoint directory, e.g. `out/checkpoint-82`.

Response shape:
```json
{
  "output": "generated text",
  "model_id": "Qwen/Qwen3-0.6B+lora",
  "latency_ms": 1234.5,
  "finish_reason": "stop"
}
```

> **Prerequisite:** the LoRA adapter must exist at `./out/adapter` (or the path passed via `?adapter=`).
> On a machine without a GPU the endpoint starts but returns a 500 when the model tries to load.

In [ ]:
%%cmd
curl -s -X POST "http://localhost:8000/v1/chat/completions?adapter=out/checkpoint-82" -H "Content-Type: application/json" -d "{\"messages\": [{\"role\": \"user\", \"content\": \"Why does the Transformer use multi-head attention?\"}], \"temperature\": 0.2}"

In [ ]:
# LoRA inference - Python
payload = {
    'messages': [{'role': 'user', 'content': DEMO_NOTES}],
    'max_new_tokens': 256,
    'temperature': 0.2,
}
r = requests.post(f'{BASE}/v1/chat/completions', json=payload, params={'adapter': ADAPTER})
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print('-- Response --')
    print(data['output'])
    print(f'\nmodel_id   : {data["model_id"]}')
    print(f'latency_ms : {data["latency_ms"]:.0f}')
    print(f'finish     : {data["finish_reason"]}')

---
## 3 · Full Raw Response
Shows the complete JSON as returned by the API - useful for debugging and benchmarking.

In [ ]:
# Full raw response dump
r = requests.post(
    f'{BASE}/v1/chat/completions',
    json={'messages': [{'role': 'user', 'content': DEMO_NOTES}]},
    params={'adapter': ADAPTER},
)
print(json.dumps(r.json(), indent=2))

---
## 4 · Failure Mode - Empty Messages List (400)

The inference route validates `InferenceRequest`, whose `messages` field has **no** `min_length`, so an empty list is not rejected by Pydantic. Instead the explicit guard in the route handler (`if not req.messages: raise HTTPException(400, "messages required")`) returns **400** before any model inference runs - no GPU cycles spent. (`TrainingExample.min_length=2` guards the *training* JSONL, a different path.)

> This is the same pattern across all weeks: validation fires first, no tokens consumed.

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/v1/chat/completions -H "Content-Type: application/json" -d "{\"messages\": []}"

In [ ]:
# Failure: empty messages - Python
r = requests.post(f'{BASE}/v1/chat/completions', json={'messages': []})
print(f'Status: {r.status_code}  (expected 400 - empty messages blocked by the route handler before model load)')
print(json.dumps(r.json(), indent=2))

---
## 5 · Failure Mode - Wrong Role Value (422)

`ChatMessage.role` is `Literal["system", "user", "assistant"]`. Sending an unknown role
returns a **422** with Pydantic's validation error - same guard that catches bad JSONL
training data (e.g. `"assistent"` typo) before it reaches the model.

> Pydantic 422s come back as an array of error objects. Use `_fmtDetail` in the browser UI
> to convert the array to a readable string.

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/v1/chat/completions -H "Content-Type: application/json" -d "{\"messages\": [{\"role\": \"assistent\", \"content\": \"hi\"}]}"

In [ ]:
# Failure: bad role literal - Python
r = requests.post(
    f'{BASE}/v1/chat/completions',
    json={'messages': [{'role': 'assistent', 'content': 'hi'}]},  # typo: 'assistent'
)
print(f'Status: {r.status_code}  (expected 422 - role literal rejected by Pydantic)')
# Pydantic 422 detail is a list of error objects
for err in r.json().get('detail', []):
    print(f'  loc: {err.get("loc")}  msg: {err.get("msg")}')

---
## 6 · OpenAPI / Swagger Docs
FastAPI auto-generates interactive docs - try endpoints live in the browser:

> This section is identical across all weeks. Do not modify it.

In [ ]:
from IPython.display import display, HTML
display(HTML('<a href="http://localhost:8000/docs" target="_blank" style="font-size:15px">'
             'Open Swagger UI: http://localhost:8000/docs</a>'))